In [5]:
"""
Modular RAG Pipeline Implementation for Fixed Token Chunking.
"""

import sys
import os
import pandas as pd
import tiktoken
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any

# ==========================================
# 1. Path Definitions and Environment Setup
# ==========================================

# Define paths dynamically
base_dir_str: str = os.path.abspath("")  
data_rel_path_str: str = os.path.join("..", "data") 
data_abs_path_str: str = os.path.normpath(os.path.join(base_dir_str, data_rel_path_str))

# Add the absolute path to sys.path to ensure module imports work correctly
if data_abs_path_str not in sys.path:
    sys.path.append(data_abs_path_str)

# Import your custom dataset
try:
    import dataset
    dataset_imported_bol: bool = True
except ImportError as e:
    print(f"Warning: Could not import dataset module from {data_abs_path_str}. Error: {e}")
    dataset_imported_bol: bool = False

In [6]:

# ==========================================
# 2. Configuration Variables
# ==========================================

pipeline_config_dct: Dict[str, Any] = {
    "chunk_size_int": 256,
    "chunk_overlap_int": 20,
    "encoding_model_str": "cl100k_base", 
    "embedding_model_str": "all-MiniLM-L6-v2", 
    "top_k_results_int": 3,
    "paths_dct": {
        "BaseAbsPath_str": base_dir_str,
        "DataRelPath_str": data_rel_path_str,
        "DataAbsPath_str": data_abs_path_str
    }
}

# ==========================================
# 3. Fixed Token Chunking Module
# ==========================================

class FixedTokenChunker:
    """
    Class to handle fixed token chunking of text documents.
    """
    def __init__(self, config_dct: Dict[str, Any]):
        """
        Initializes the chunker with encoding model and parameters.
        
        Args:
            config_dct (Dict[str, Any]): Configuration dictionary containing chunk size, 
                                         overlap, and encoding model.
        """
        self.chunk_size_int: int = config_dct.get("chunk_size_int", 100)
        self.chunk_overlap_int: int = config_dct.get("chunk_overlap_int", 10)
        self.encoding_model_str: str = config_dct.get("encoding_model_str", "cl100k_base")
        self.tokenizer_ins = tiktoken.get_encoding(self.encoding_model_str)

    def chunk_text(self, text_str: str) -> Dict[str, Any]:
        """
        Splits text into fixed token chunks with an overlap.
        
        Args:
            text_str (str): The input string to be chunked.
            
        Returns:
            Dict[str, Any]: A result dictionary containing the list of text chunks, 
                            the token counts, and execution status.
        """
        result_dct: Dict[str, Any] = {
            "chunks_lst": [],
            "token_counts_lst": [],
            "status_bol": False,
            "error_message_str": ""
        }
        
        try:
            tokens_lst: List[int] = self.tokenizer_ins.encode(text_str)
            chunks_text_lst: List[str] = []
            token_counts_lst: List[int] = []
            
            # Step size for loop
            step_int: int = self.chunk_size_int - self.chunk_overlap_int
            
            for i_int in range(0, len(tokens_lst), step_int):
                chunk_tokens_lst: List[int] = tokens_lst[i_int: i_int + self.chunk_size_int]
                chunk_text_str: str = self.tokenizer_ins.decode(chunk_tokens_lst)
                
                chunks_text_lst.append(chunk_text_str)
                token_counts_lst.append(len(chunk_tokens_lst))
                
            result_dct["chunks_lst"] = chunks_text_lst
            result_dct["token_counts_lst"] = token_counts_lst
            result_dct["status_bol"] = True
            
        except Exception as e:
            result_dct["error_message_str"] = str(e)
            
        return result_dct

In [7]:
# ==========================================
# 4. Embedding and Retrieval Module
# ==========================================

class VectorStoreRetrieval:
    """
    Class to embed document chunks and retrieve them based on vector similarity.
    """
    def __init__(self, config_dct: Dict[str, Any]):
        """
        Initializes the embedding model and vector storage.
        
        Args:
            config_dct (Dict[str, Any]): Configuration dictionary containing the embedding model name.
        """
        self.model_name_str: str = config_dct.get("embedding_model_str", "all-MiniLM-L6-v2")
        self.embedder_ins = SentenceTransformer(self.model_name_str)
        self.corpus_embeddings_lst: List[Any] = []
        self.knowledge_base_df: pd.DataFrame = pd.DataFrame()
        
    def index_documents(self, chunks_df: pd.DataFrame) -> Dict[str, Any]:
        """
        Embeds the document chunks and stores them.
        
        Args:
            chunks_df (pd.DataFrame): DataFrame containing a 'text_chunk_str' column.
            
        Returns:
            Dict[str, Any]: Result dictionary containing status and number of indexed chunks.
        """
        result_dct: Dict[str, Any] = {
            "indexed_count_int": 0,
            "status_bol": False,
            "error_message_str": ""
        }
        
        try:
            text_chunks_lst: List[str] = chunks_df['text_chunk_str'].tolist()
            # Generate embeddings
            embeddings_arr = self.embedder_ins.encode(text_chunks_lst)
            
            self.corpus_embeddings_lst = embeddings_arr
            self.knowledge_base_df = chunks_df
            
            result_dct["indexed_count_int"] = len(text_chunks_lst)
            result_dct["status_bol"] = True
            
        except Exception as e:
            result_dct["error_message_str"] = str(e)
            
        return result_dct
        
    def retrieve_context(self, query_str: str, top_k_int: int = 3) -> Dict[str, Any]:
        """
        Retrieves the top k most similar chunks for a given query.
        
        Args:
            query_str (str): The user query.
            top_k_int (int): The number of top results to return.
            
        Returns:
            Dict[str, Any]: Result dictionary containing the retrieved contexts and scores.
        """
        result_dct: Dict[str, Any] = {
            "retrieved_contexts_lst": [],
            "scores_lst": [],
            "status_bol": False,
            "error_message_str": ""
        }
        
        try:
            query_embedding_arr = self.embedder_ins.encode([query_str])
            similarity_scores_arr = cosine_similarity(query_embedding_arr, self.corpus_embeddings_lst)[0]
            
            # Get indices of top k scores
            top_k_indices_arr = np.argsort(similarity_scores_arr)[::-1][:top_k_int]
            
            contexts_lst: List[str] = [self.knowledge_base_df.iloc[i]['text_chunk_str'] for i in top_k_indices_arr]
            scores_lst: List[float] = [similarity_scores_arr[i] for i in top_k_indices_arr]
            
            result_dct["retrieved_contexts_lst"] = contexts_lst
            result_dct["scores_lst"] = scores_lst
            result_dct["status_bol"] = True
            
        except Exception as e:
            result_dct["error_message_str"] = str(e)
            
        return result_dct

In [8]:

# ==========================================
# 5. Pipeline Execution 
# ==========================================

def run_pipeline() -> None:
    """
    Executes the end-to-end chunking and retrieval pipeline.
    """
    # 1. Load Data
    if dataset_imported_bol and hasattr(dataset, 'test'):
        raw_documents_lst: List[str] = [dataset.test]
        print("Using documents from dataset.test module.")
    else:
        raw_documents_lst: List[str] = [
            "Machine learning is a subset of artificial intelligence that uses statistical models.",
            "Retrieval-Augmented Generation (RAG) systems combine vector search with LLMs for better AI responses.",
            "Python is a high-level programming language known for its readability and versatility."
        ]
        print("Using fallback dummy documents.")

    # 2. Instantiate classes
    print("Initializing components...")
    chunker_ins = FixedTokenChunker(config_dct=pipeline_config_dct)
    retriever_ins = VectorStoreRetrieval(config_dct=pipeline_config_dct)

    # 3. Process documents and collect chunks
    print("Chunking documents...")
    all_chunks_data_lst: List[Dict[str, Any]] = []

    for doc_idx_int, document_str in enumerate(raw_documents_lst):
        chunking_results_dct: Dict[str, Any] = chunker_ins.chunk_text(text_str=document_str)
        
        if chunking_results_dct["status_bol"]:
            for chunk_idx_int, chunk_text_str in enumerate(chunking_results_dct["chunks_lst"]):
                all_chunks_data_lst.append({
                    "doc_id_int": doc_idx_int,
                    "chunk_id_int": chunk_idx_int,
                    "text_chunk_str": chunk_text_str
                })
        else:
            print(f"Failed to chunk document {doc_idx_int}: {chunking_results_dct['error_message_str']}")

    # 4. Create proper Dataframe
    document_chunks_df: pd.DataFrame = pd.DataFrame(all_chunks_data_lst)
    print(f"Total chunks created: {len(document_chunks_df)}")

    # 5. Index chunks into Vector Store
    print("Indexing documents into vector store...")
    indexing_results_dct: Dict[str, Any] = retriever_ins.index_documents(chunks_df=document_chunks_df)
    
    if indexing_results_dct["status_bol"]:
        print(f"Successfully indexed {indexing_results_dct['indexed_count_int']} chunks.")
    else:
        print(f"Indexing Failed: {indexing_results_dct['error_message_str']}")
        return

    # 6. Test Retrieval
    test_query_str: str = "What is clonal hematopoiesis and how common is it in elderly individuals?"
    print(f"\nTesting Retrieval with query: '{test_query_str}'")
    
    retrieval_results_dct: Dict[str, Any] = retriever_ins.retrieve_context(
        query_str=test_query_str, 
        top_k_int=pipeline_config_dct["top_k_results_int"]
    )

    if retrieval_results_dct["status_bol"]:
        for idx_int, context_str in enumerate(retrieval_results_dct["retrieved_contexts_lst"]):
            score_flt: float = retrieval_results_dct["scores_lst"][idx_int]
            print(f"Result {idx_int + 1} (Similarity Score: {score_flt:.4f}):\n{context_str}\n")
    else:
        print(f"Retrieval Error: {retrieval_results_dct['error_message_str']}")

if __name__ == "__main__":
    run_pipeline()

Using documents from dataset.test module.
Initializing components...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 629.01it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunking documents...
Total chunks created: 23
Indexing documents into vector store...
Successfully indexed 23 chunks.

Testing Retrieval with query: 'What is clonal hematopoiesis and how common is it in elderly individuals?'
Result 1 (Similarity Score: 0.5945):
 the surprisingly high numbers of participants with 3 to 18 detectable mutations. In our analyses below, we classified 195 participants as having clonal hematopoiesis with unknown drivers. In some cases of clonal hematopoiesis with unknown drivers, additional analysis suggested potential candidate drivers (Fig. S12 in the Supplementary Appendix).
Clonal Hematopoiesis and Advancing Age
Detectable clonal hematopoiesis with candidate drivers was rare among younger participants (occurring in 0.7% of participants younger than 50 years of age) but much more common among older participants (occurring in 5.7% of participants older than 65 years of age) (Figure 2D). Reflecting this relationship, participants having clonal hematopoiesis 